In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
df = pd.read_csv("sales_pos.csv")
df.head(2)

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase
0,1,P00069042,F,0-17,10,A,0,3,NaN,NaN,8370
1,1,P00248942,F,0-17,10,A,0,1,6.0,14.0,15200


### Q1.

In [5]:
ser_g = df.groupby("prod")["purchase"].sum()
ser_g[ser_g == ser_g.max()]

prod
P00025442    27995166
Name: purchase, dtype: int64

In [7]:
val_top_prod = ser_g.idxmax()
val_top_prod

'P00025442'

In [11]:
df.loc[df["prod"] == val_top_prod, "job"].value_counts().idxmax()

4

### Q2.

In [ ]:
df_u1 = df.loc[df["user"] == 1, ["prod_cat1", "prod_cat2", "prod_cat3"]].reset_index(drop = True)
df_u1 = df_u1.fillna(0)
df_u1.head(2)

In [ ]:
len(df_u1)

In [ ]:
df_u1

In [ ]:
df_u1.sort_values(df_u1.columns.to_list())

In [ ]:
df_u1.drop_duplicates().shape # 1

In [23]:
df_u1["prod_cat1"] = df_u1["prod_cat1"].astype("str")
df_u1["prod_cat2"] = df_u1["prod_cat2"].astype("int").astype("str") # .astype("int") 는 없어도 정답산출에 지장 없음.
df_u1["prod_cat3"] = df_u1["prod_cat3"].astype("int").astype("str")
df_u1["prod_cat"] = df_u1["prod_cat1"] + "-" + df_u1["prod_cat2"] + "-" + df_u1["prod_cat3"]

In [25]:
df_u1.head(1)

,prod_cat1,prod_cat2,prod_cat3,prod_cat
0,3,0,0,3-0-0


In [26]:
df_u1["prod_cat"].nunique() # 2

21

In [34]:
df_q2 = df.loc[df["age_group"] == "26-35", ["user", "marital", "prod_cat1", "prod_cat2", "prod_cat3"]]
df_q2 = df_q2.reset_index(drop = True)
df_q2 = df_q2.fillna(0)
df_q2["prod_cat1"] = df_q2["prod_cat1"].astype("str")
df_q2["prod_cat2"] = df_q2["prod_cat2"].astype("int").astype("str")
df_q2["prod_cat3"] = df_q2["prod_cat3"].astype("int").astype("str")
df_q2["prod_cat"]  = df_q2["prod_cat1"] + "-" + df_q2["prod_cat2"] + "-" + df_q2["prod_cat3"]

In [ ]:
df_q2_g = df_q2.groupby(["user", "marital"])["prod_cat"].nunique().reset_index()
df_q2_g.head(2)

In [38]:
stat_m0 = df_q2_g.loc[df_q2_g["marital"] == 0, "prod_cat"].mean()
stat_m1 = df_q2_g.loc[df_q2_g["marital"] == 1, "prod_cat"].mean()
stat_m0, stat_m1

(41.66318327974277, 41.79233621755253)

In [39]:
round(abs(stat_m0 - stat_m1), 2)

0.13

### Q3.

In [ ]:
df_user = df[["user", "gender", "age_group", "job", "city", "marital"]].drop_duplicates()
df_user.head(1)

In [ ]:
df_g = df.groupby("user")[["prod", "purchase"]].agg({"prod": "nunique", "purchase": "sum"})
df_g = df_g.reset_index()
df_g.head(1)

In [ ]:
df_join = pd.merge(df_user, df_g, on = "user", how = "inner")
df_join.head(2) # 1

In [51]:
ls_col_g = ["user", "gender", "age_group", "job", "city", "marital"]
df_join = df.groupby(ls_col_g)[["prod", "purchase"]].agg({"prod": "nunique", "purchase": "sum"})
df_join = df_join.reset_index()
df_join.head(1) # 2

,user,gender,age_group,job,city,marital,prod,purchase
0,1,F,0-17,10,A,0,35,334093


문제 설계가 미흡하여 1번과 2번 접근은 각각의 결과가 다름. 이 부분을 해결하려면 이 다음에 "user"변수 기준으로 정렬하라는 지시사항이 추가되어야 함.

In [ ]:
df_join["gender"] = df_join["gender"].replace({"M": 1, "F": 0})

In [ ]:
ser_u = df_join["age_group"].drop_duplicates().sort_values()
ser_repl = pd.Series(range(len(ser_u)), index = ser_u)
ser_repl

In [ ]:
df_join["age_group"] = df_join["age_group"].replace(ser_repl)

In [ ]:
# df_join_dum = pd.get_dummies(df_join, columns = ["job", "city"]) # 시험버전
df_join_dum = pd.get_dummies(df_join, columns = ["job", "city"], dtype = "int") # 최신버전
df_join_dum = df_join_dum.drop(columns = "user")
df_join_dum.head(1)

In [ ]:
arr_model_nor = MinMaxScaler().fit_transform(df_join_dum)
arr_model_nor[:1, ]

In [ ]:
model_kmeans = KMeans(n_clusters = 7, random_state = 123)
model_kmeans.fit(arr_model_nor)

In [66]:
round(silhouette_score(arr_model_nor, labels = model_kmeans.labels_), 2)

0.18